# S&P 500 Volatility Forecasting - Exploratory Analysis

This notebook walks through the full pipeline step-by-step.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from config import TRAIN_CUTOFF
from src.data_loader import load_shiller_data, calculate_returns_and_volatility, clean_data, temporal_split
from src.features import engineer_features, get_feature_columns
from src.models import NaiveModel, RidgeModel, RandomForestModel, XGBoostModel, LSTMModel
from src.ensemble import WeightedEnsemble, EqualEnsemble
from src.evaluation import evaluate, print_results
from src.visualization import plot_results, plot_feature_importance

plt.style.use('seaborn-v0_8-whitegrid')
print("Libraries loaded!")

## 1. Load Data

In [ ]:
raw_df = load_shiller_data()
print(f"Loaded {len(raw_df)} months from {raw_df['Date'].min().date()} to {raw_df['Date'].max().date()}")
raw_df[['Date', 'SP500', 'PE10', 'Long Interest Rate']].head()

## 2. Calculate Returns & Volatility

In [ ]:
df = calculate_returns_and_volatility(raw_df)
df = clean_data(df)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(df['Date'], df['SP500'], color='#2c3e50')
axes[0].set_title('S&P 500 Index (1950-2026)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Index Value')

axes[1].plot(df['Date'], df['realized_vol_6m'], color='#e74c3c')
axes[1].set_title('6-Month Realized Volatility (Annualized)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Volatility')
axes[1].set_xlabel('Year')

plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
features_df = engineer_features(df)
feature_cols = get_feature_columns(features_df)
print(f"Features: {feature_cols}")
print(f"\nDataset shape: {features_df.shape}")
features_df.head()

## 4. Train/Test Split

In [ ]:
train_df, test_df = temporal_split(features_df, TRAIN_CUTOFF)
X_train, y_train = train_df[feature_cols].values, train_df['target'].values
X_test, y_test = test_df[feature_cols].values, test_df['target'].values

print(f"Train: {len(X_train)} samples ({train_df['Date'].min().date()} to {train_df['Date'].max().date()})")
print(f"Test:  {len(X_test)} samples ({test_df['Date'].min().date()} to {test_df['Date'].max().date()})")

## 5. Train Models

In [ ]:
# Train all models
naive = NaiveModel()
ridge = RidgeModel(alpha=1.0).fit(X_train, y_train)
rf = RandomForestModel().fit(X_train, y_train)
xgb = XGBoostModel().fit(X_train, y_train)
lstm = LSTMModel().fit(X_train, y_train)

# Predictions
y_naive = naive.predict(X_test, features_df)
y_ridge = ridge.predict(X_test)
y_rf = rf.predict(X_test)
y_xgb = xgb.predict(X_test)
y_lstm = lstm.predict(X_test)

print("All models trained and predictions generated!")

## 6. Ensemble

In [ ]:
val_size = int(0.2 * len(X_train))
X_val, y_val = X_train[-val_size:], y_train[-val_size:]

# Fit validation models
models_val = {
    'Ridge': RidgeModel(alpha=1.0).fit(X_train[:-val_size], y_train[:-val_size]),
    'Random Forest': RandomForestModel().fit(X_train[:-val_size], y_train[:-val_size]),
    'XGBoost': XGBoostModel().fit(X_train[:-val_size], y_train[:-val_size]),
    'LSTM': LSTMModel().fit(X_train[:-val_size], y_train[:-val_size])
}

weighted_ens = WeightedEnsemble(models_val)
weighted_ens.fit_weights(X_val, y_val)
y_ensemble = weighted_ens.predict(X_test)

equal_ens = EqualEnsemble({'Ridge': ridge, 'RF': rf, 'XGB': xgb, 'LSTM': lstm})
y_equal = equal_ens.predict(X_test)

## 7. Evaluate

In [ ]:
results = []
results.append(evaluate(y_test, y_naive, 'Naive'))
results.append(evaluate(y_test, y_ridge, 'Ridge'))
results.append(evaluate(y_test, y_rf, 'Random Forest'))
results.append(evaluate(y_test, y_xgb, 'XGBoost'))
results.append(evaluate(y_test, y_lstm, 'LSTM'))
results.append(evaluate(y_test, y_equal, 'Ensemble (Equal)'))
results.append(evaluate(y_test, y_ensemble, 'Ensemble (Weighted)'))

print_results(results)

## 8. Visualize

In [ ]:
test_dates = test_df['Date'].values
predictions_dict = {
    'Random Forest': y_rf,
    'XGBoost': y_xgb,
    'Weighted Ensemble': y_ensemble,
    'Naive': y_naive
}

plot_results(test_dates, y_test, predictions_dict)
plot_feature_importance(feature_cols, xgb.feature_importances())